In [ ]:
# ============================================================
# SU(2) 4D DRIFT: BETA-SWEEP (single-pass over MC directions)
#
# Goal:
#   For a fixed dataset U ~ (sigma-mixture + Haar),
#   compute per-config (Vbar, Bavg, lap) and for each beta in BETA_LIST:
#     LV_beta = lap - <grad S_beta, grad V>
#   then fit and holdout-check LV_beta <= -lambda V + b (one-sided).
#
# Notes:
#   - Heavy ops (plaquettes on Up/Um) are beta-independent and amortized.
#   - U is stored as quaternions: (...,4) with q=(a,b,c,d).
#   - Uses float64 for U / accumulators, float32 for Xi.
# ============================================================

import os, math, time
import numpy as np
import torch

# ----------------------------
# USER CONFIG
# ----------------------------
CFG = dict(
    L=12,
    K_total=2048,
    K_fit=1024,

    eps_fd=5e-3,
    mc=256,
    mc_chunk=1,            # memory knob; 1 is safe (you used it successfully)
    batch_cfg=64,          # stream configs in batches (memory knob)

    sigma_list=[0.0, 0.1, 0.2, 0.4, 0.8, 1.6],
    frac_haar=0.25,

    beta_list=[2.0, 4.0, 6.0, 8.0, 10.0],   # edit freely

    seed=1234,
    out_npz="beta_sweep_results.npz",
    n_grid=4001,           # lambda grid for one-sided fit
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device} {torch.cuda.get_device_name(0) if device.type=='cuda' else ''}", flush=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

U_dtype  = torch.float64
Xi_dtype = torch.float32

# ----------------------------
# SU(2) quaternion ops
# ----------------------------
def su2_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return q / (q.pow(2).sum(dim=-1, keepdim=True).clamp_min(eps).sqrt())

def su2_mul(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
    a1, b1, c1, d1 = q1.unbind(-1)
    a2, b2, c2, d2 = q2.unbind(-1)
    return torch.stack(
        [
            a1*a2 - b1*b2 - c1*c2 - d1*d2,
            a1*b2 + b1*a2 + c1*d2 - d1*c2,
            a1*c2 - b1*d2 + c1*a2 + d1*b2,
            a1*d2 + b1*c2 - c1*b2 + d1*a2,
        ],
        dim=-1
    )

def su2_conj(q: torch.Tensor) -> torch.Tensor:
    a, b, c, d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v: torch.Tensor) -> torch.Tensor:
    # v: (...,3)
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    a = torch.cos(theta)
    theta2 = theta * theta
    small = theta < 1e-4
    s_over_theta = torch.where(
        small,
        1.0 - theta2/6.0 + (theta2*theta2)/120.0,
        torch.sin(theta) / theta.clamp_min(1e-30),
    )
    vec = s_over_theta * v
    return torch.cat([a, vec], dim=-1)

# ----------------------------
# Plaquette defect sums (memory-light)
# Returns zsum (sum over plaq of (1 - a_p)), and Vbar = 1 + Bavg
# ----------------------------
@torch.no_grad()
def zsum_Bavg_Vbar(U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # U: (B, L,L,L,L, 4, 4)
    B, L = U.shape[0], U.shape[1]
    pairs = [(mu, nu) for mu in range(4) for nu in range(mu+1, 4)]
    zsum = torch.zeros((B,), device=U.device, dtype=U_dtype)

    for mu, nu in pairs:
        U_mu = U[..., mu, :]
        U_nu = U[..., nu, :]
        U_nu_xpmu = torch.roll(U_nu, shifts=-1, dims=1 + mu)
        U_mu_xpnu = torch.roll(U_mu, shifts=-1, dims=1 + nu)

        # P = U_mu(x) U_nu(x+mu) U_mu(x+nu)^dag U_nu(x)^dag
        P = su2_mul(su2_mul(su2_mul(U_mu, U_nu_xpmu), su2_conj(U_mu_xpnu)), su2_conj(U_nu))
        defect = 1.0 - P[..., 0]  # (B,L,L,L,L)
        zsum += defect.sum(dim=(1,2,3,4))

    total = len(pairs) * (L**4)
    Bavg = zsum / float(total)
    Vbar = 1.0 + Bavg
    return zsum, Bavg, Vbar

# ----------------------------
# Dataset meta + generator (sigma-mixture + Haar)
# ----------------------------
def make_meta(K_total: int, sigma_list: list[float], frac_haar: float, seed: int):
    K_haar = int(round(frac_haar * K_total))
    K_sig  = K_total - K_haar
    per = K_sig // len(sigma_list)
    leftover = K_sig - per * len(sigma_list)

    meta = []
    for i, s in enumerate(sigma_list):
        cnt = per + (1 if i < leftover else 0)
        meta += [("sigma", float(s))] * cnt
    meta += [("haar", float("nan"))] * K_haar

    rng = np.random.default_rng(seed)
    rng.shuffle(meta)
    return meta

@torch.no_grad()
def gen_U_batch(meta_batch, L: int):
    B = len(meta_batch)
    U = torch.empty((B, L, L, L, L, 4, 4), device=device, dtype=U_dtype)

    kinds = [k for k, _ in meta_batch]
    sigs  = [s for _, s in meta_batch]
    idx_sigma = [i for i, k in enumerate(kinds) if k == "sigma"]
    idx_haar  = [i for i, k in enumerate(kinds) if k == "haar"]

    if idx_sigma:
        sigma = torch.tensor([sigs[i] for i in idx_sigma], device=device, dtype=Xi_dtype).view(-1,1,1,1,1,1,1)
        Xi = torch.randn((len(idx_sigma), L, L, L, L, 4, 3), device=device, dtype=Xi_dtype)
        U[idx_sigma] = su2_exp(sigma * Xi).to(dtype=U_dtype)

    if idx_haar:
        q = torch.randn((len(idx_haar), L, L, L, L, 4, 4), device=device, dtype=U_dtype)
        U[idx_haar] = su2_normalize(q)

    return U

# ----------------------------
# One-sided drift fit: LV + margin*SE <= -lambda V + b
# ----------------------------
@torch.no_grad()
def fit_drift_one_sided(V_fit, LV_fit, SE_fit, margin_sigma: float, n_grid: int):
    # scan lambda >= 0
    X = torch.stack([V_fit, torch.ones_like(V_fit)], dim=1)
    sol = torch.linalg.lstsq(X, LV_fit.unsqueeze(1)).solution.squeeze()
    a = float(sol[0].item())
    lam_ols = max(0.0, float(-a))
    lam_max = max(5.0, 5.0*(lam_ols + 1.0))

    lambdas = torch.linspace(0.0, lam_max, n_grid, device=V_fit.device, dtype=V_fit.dtype)
    rhs = (LV_fit + margin_sigma*SE_fit).unsqueeze(0) + lambdas.unsqueeze(1) * V_fit.unsqueeze(0)
    b_vals = rhs.max(dim=1).values
    j = int(torch.argmin(b_vals).item())
    return float(lambdas[j].item()), float(b_vals[j].item())

@torch.no_grad()
def holdout_violations(V, LV, SE, lam, b, ksig=(0.0,2.0,5.0)):
    out = {}
    for k in ksig:
        diff = (LV + k*SE) + lam*V - b
        viol = diff > 0.0
        out[k] = dict(
            violations=int(viol.sum().item()),
            total=int(V.numel()),
            frac=float(viol.float().mean().item()),
            worst=float(diff.max().item()),
        )
    return out

# ----------------------------
# Main: beta-sweep in one pass
# ----------------------------
@torch.no_grad()
def run():
    L = int(CFG["L"])
    K_total = int(CFG["K_total"])
    K_fit = int(CFG["K_fit"])
    eps = float(CFG["eps_fd"])
    mc = int(CFG["mc"])
    mc_chunk = int(CFG["mc_chunk"])
    batch_cfg = int(CFG["batch_cfg"])
    beta_list = [float(b) for b in CFG["beta_list"]]
    n_grid = int(CFG["n_grid"])

    meta = make_meta(K_total, CFG["sigma_list"], CFG["frac_haar"], CFG["seed"])

    # storage (on CPU to be safe)
    Vbar_all = np.empty((K_total,), dtype=np.float64)
    Bavg_all = np.empty((K_total,), dtype=np.float64)
    lap_all  = np.empty((K_total,), dtype=np.float64)
    lap_se_all = np.empty((K_total,), dtype=np.float64)

    LV_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    LV_se_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    gip_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    gip_se_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}

    t0 = time.time()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # stream configs
    for start in range(0, K_total, batch_cfg):
        end = min(K_total, start + batch_cfg)
        mb = meta[start:end]
        U = gen_U_batch(mb, L=L)

        # base observables on unperturbed U
        z0, Bavg0, V0 = zsum_Bavg_Vbar(U)

        B = U.shape[0]
        # accumulators for lap and (z_p - z_m) * dV (beta-independent pieces)
        sum_lap  = torch.zeros((B,), device=device, dtype=torch.float64)
        sum_lap2 = torch.zeros((B,), device=device, dtype=torch.float64)

        # for each beta we accumulate LV and gip moments
        sum_gip  = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_gip2 = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_LV   = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_LV2  = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}

        done = 0
        while done < mc:
            k = min(mc_chunk, mc - done)

            # Xi: (k,B,L,L,L,L,4,3)
            Xi = torch.randn((k, B, L, L, L, L, 4, 3), device=device, dtype=Xi_dtype)

            exp_p = su2_exp(eps * Xi)              # (k,B,...,4)
            exp_m = su2_conj(exp_p)

            Up = su2_mul(U.unsqueeze(0), exp_p).reshape(k*B, L, L, L, L, 4, 4)
            Um = su2_mul(U.unsqueeze(0), exp_m).reshape(k*B, L, L, L, L, 4, 4)

            z_p, _, V_p = zsum_Bavg_Vbar(Up)
            z_m, _, V_m = zsum_Bavg_Vbar(Um)

            z_p = z_p.view(k, B)
            z_m = z_m.view(k, B)
            V_p = V_p.view(k, B)
            V_m = V_m.view(k, B)

            lap = (V_p + V_m - 2.0*V0.unsqueeze(0)) / (eps*eps)   # (k,B)
            dV  = (V_p - V_m) / (2.0*eps)                         # (k,B)
            dz  = (z_p - z_m) / (2.0*eps)                         # (k,B)  (beta-free)

            # accumulate lap moments
            sum_lap  += lap.sum(dim=0).to(torch.float64)
            sum_lap2 += (lap*lap).sum(dim=0).to(torch.float64)

            # now do all betas cheaply
            for beta in beta_list:
                dS = beta * dz                # (k,B)
                gip = dS * dV                 # (k,B)
                LV  = lap - gip               # (k,B)

                sum_gip[beta]  += gip.sum(dim=0).to(torch.float64)
                sum_gip2[beta] += (gip*gip).sum(dim=0).to(torch.float64)
                sum_LV[beta]   += LV.sum(dim=0).to(torch.float64)
                sum_LV2[beta]  += (LV*LV).sum(dim=0).to(torch.float64)

            done += k
            del Xi, exp_p, exp_m, Up, Um, z_p, z_m, V_p, V_m, lap, dV, dz
            if device.type == "cuda":
                torch.cuda.empty_cache()

        # finalize mean + SE (per-config) for this batch
        def mean_se(sum1, sum2, n):
            mean = sum1 / float(n)
            var = (sum2 - (sum1*sum1)/float(n)) / float(n-1)
            var = torch.clamp(var, min=0.0)
            se = torch.sqrt(var / float(n))
            return mean, se

        lap_m, lap_se = mean_se(sum_lap, sum_lap2, mc)

        # write to CPU arrays
        Vbar_all[start:end] = V0.detach().cpu().numpy()
        Bavg_all[start:end] = Bavg0.detach().cpu().numpy()
        lap_all[start:end]  = lap_m.detach().cpu().numpy()
        lap_se_all[start:end] = lap_se.detach().cpu().numpy()

        for beta in beta_list:
            gip_m, gip_se = mean_se(sum_gip[beta], sum_gip2[beta], mc)
            LV_m, LV_se = mean_se(sum_LV[beta], sum_LV2[beta], mc)
            gip_all[beta][start:end] = gip_m.detach().cpu().numpy()
            gip_se_all[beta][start:end] = gip_se.detach().cpu().numpy()
            LV_all[beta][start:end] = LV_m.detach().cpu().numpy()
            LV_se_all[beta][start:end] = LV_se.detach().cpu().numpy()

        del U, z0, Bavg0, V0, sum_lap, sum_lap2
        if device.type == "cuda":
            torch.cuda.empty_cache()

        rate = (end) / max(1e-9, (time.time()-t0))
        print(f"[progress] {end}/{K_total} configs  ({rate:.2f} cfg/s)", flush=True)

    # ----------------------------
    # Fit + holdout per beta
    # ----------------------------
    print("\n============================")
    print("BETA-SWEEP FIT + HOLDOUT")
    print("============================")
    V = torch.tensor(Vbar_all, dtype=torch.float64)
    for beta in beta_list:
        LV = torch.tensor(LV_all[beta], dtype=torch.float64)
        SE = torch.tensor(LV_se_all[beta], dtype=torch.float64)

        V_fit, LV_fit, SE_fit = V[:K_fit], LV[:K_fit], SE[:K_fit]
        V_te,  LV_te,  SE_te  = V[K_fit:], LV[K_fit:], SE[K_fit:]

        lam0, b0


[device] cpu 


In [ ]:
# ============================================================
# SU(2) 4D DRIFT — SINGLE-PASS BETA SWEEP
#
# What this does:
#   Generates a dataset of SU(2) link fields U on a 4D L^4 torus
#   (mixture of Haar + exp(sigma * N(0,1))).
#
#   For each configuration, estimates:
#     Vbar = 1 + average_plaquette_defect
#     lap  = finite-difference Laplacian of Vbar along random tangent directions
#
#   For each beta in BETA_LIST, estimates:
#     gip_beta = <grad S_beta, grad Vbar> via symmetric finite-differences
#     LV_beta  = lap - gip_beta
#
#   Then fits a one-sided drift inequality on a fit set:
#     LV + k*SE <= -lambda * V + b
#   and reports holdout violations.
#
# Output:
#   Saves a compressed NPZ with arrays:
#     Vbar, Bavg, lap, lap_se, and per-beta LV/gip + SEs
#
# Compute knobs:
#   L, K_total, mc, mc_chunk, batch_cfg, eps_fd
# ============================================================

import os, time, math
import numpy as np
import torch

# ----------------------------
# CONFIG
# ----------------------------
CFG = dict(
    L=12,                 # lattice linear size
    K_total=2048,          # number of configs
    K_fit=1024,            # fit / holdout split

    eps_fd=5e-3,           # finite-difference step
    mc=256,                # MC directions per config
    mc_chunk=1,            # safe default; increase only if you know memory is fine
    batch_cfg=64,          # number of configs per batch

    sigma_list=[0.0, 0.1, 0.2, 0.4, 0.8, 1.6],
    frac_haar=0.25,

    beta_list=[2.0, 4.0, 6.0, 8.0, 10.0],

    seed=1234,
    out_npz="beta_sweep_results.npz",

    n_grid=4001,           # lambda grid points for one-sided fit scan
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device} {torch.cuda.get_device_name(0) if device.type=='cuda' else ''}", flush=True)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

U_dtype  = torch.float64
Xi_dtype = torch.float32

# ============================================================
# SU(2) quaternion algebra
# q = (a,b,c,d) representing a*I + i(b sigma1 + c sigma2 + d sigma3)
# ============================================================

def su2_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return q / (q.pow(2).sum(dim=-1, keepdim=True).clamp_min(eps).sqrt())

def su2_conj(q: torch.Tensor) -> torch.Tensor:
    a, b, c, d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_mul(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
    a1, b1, c1, d1 = q1.unbind(-1)
    a2, b2, c2, d2 = q2.unbind(-1)
    return torch.stack(
        [
            a1*a2 - b1*b2 - c1*c2 - d1*d2,
            a1*b2 + b1*a2 + c1*d2 - d1*c2,
            a1*c2 - b1*d2 + c1*a2 + d1*b2,
            a1*d2 + b1*c2 - c1*b2 + d1*a2,
        ],
        dim=-1
    )

def su2_exp(v: torch.Tensor) -> torch.Tensor:
    # v: (...,3) in su(2) coordinates
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    a = torch.cos(theta)
    theta2 = theta * theta
    small = theta < 1e-4
    s_over_theta = torch.where(
        small,
        1.0 - theta2/6.0 + (theta2*theta2)/120.0,
        torch.sin(theta) / theta.clamp_min(1e-30),
    )
    vec = s_over_theta * v
    return torch.cat([a, vec], dim=-1)

# ============================================================
# Plaquette sums
# Returns:
#   zsum = sum_{plaquettes} (1 - a_p)
#   Bavg = zsum / (#plaquettes)
#   Vbar = 1 + Bavg
# ============================================================

@torch.no_grad()
def zsum_Bavg_Vbar(U: torch.Tensor):
    # U: (B, L,L,L,L, 4, 4)
    B, L = U.shape[0], U.shape[1]
    pairs = [(mu, nu) for mu in range(4) for nu in range(mu+1, 4)]
    zsum = torch.zeros((B,), device=U.device, dtype=U_dtype)

    # dims for roll: 0=batch, 1..4 = x0..x3, 5=mu, 6=quat
    for mu, nu in pairs:
        U_mu = U[..., mu, :]
        U_nu = U[..., nu, :]
        U_nu_xpmu = torch.roll(U_nu, shifts=-1, dims=1 + mu)
        U_mu_xpnu = torch.roll(U_mu, shifts=-1, dims=1 + nu)

        P = su2_mul(su2_mul(su2_mul(U_mu, U_nu_xpmu), su2_conj(U_mu_xpnu)), su2_conj(U_nu))
        defect = 1.0 - P[..., 0]  # scalar part a_p
        zsum += defect.sum(dim=(1,2,3,4))

    total = len(pairs) * (L**4)
    Bavg = zsum / float(total)
    Vbar = 1.0 + Bavg
    return zsum, Bavg, Vbar

# ============================================================
# Dataset metadata + batch generator
# ============================================================

def make_meta(K_total: int, sigma_list, frac_haar: float, seed: int):
    K_haar = int(round(frac_haar * K_total))
    K_sig  = K_total - K_haar
    per = K_sig // len(sigma_list)
    leftover = K_sig - per * len(sigma_list)

    meta = []
    for i, s in enumerate(sigma_list):
        cnt = per + (1 if i < leftover else 0)
        meta += [("sigma", float(s))] * cnt
    meta += [("haar", float("nan"))] * K_haar

    rng = np.random.default_rng(seed)
    rng.shuffle(meta)
    return meta

@torch.no_grad()
def gen_U_batch(meta_batch, L: int):
    B = len(meta_batch)
    U = torch.empty((B, L, L, L, L, 4, 4), device=device, dtype=U_dtype)

    kinds = [k for k, _ in meta_batch]
    sigs  = [s for _, s in meta_batch]
    idx_sigma = [i for i, k in enumerate(kinds) if k == "sigma"]
    idx_haar  = [i for i, k in enumerate(kinds) if k == "haar"]

    if idx_sigma:
        sigma = torch.tensor([sigs[i] for i in idx_sigma], device=device, dtype=Xi_dtype).view(-1,1,1,1,1,1,1)
        Xi = torch.randn((len(idx_sigma), L, L, L, L, 4, 3), device=device, dtype=Xi_dtype)
        U[idx_sigma] = su2_exp(sigma * Xi).to(dtype=U_dtype)

    if idx_haar:
        q = torch.randn((len(idx_haar), L, L, L, L, 4, 4), device=device, dtype=U_dtype)
        U[idx_haar] = su2_normalize(q)

    return U

# ============================================================
# One-sided drift fit + holdout
# ============================================================

@torch.no_grad()
def mean_se(sum1, sum2, n: int):
    mean = sum1 / float(n)
    var = (sum2 - (sum1*sum1)/float(n)) / float(max(1, n-1))
    var = torch.clamp(var, min=0.0)
    se = torch.sqrt(var / float(n))
    return mean, se

@torch.no_grad()
def fit_drift_one_sided(V_fit, LV_fit, SE_fit, margin_sigma: float, n_grid: int):
    # quick OLS slope guess for scan range
    X = torch.stack([V_fit, torch.ones_like(V_fit)], dim=1)
    sol = torch.linalg.lstsq(X, LV_fit.unsqueeze(1)).solution.squeeze()
    a = float(sol[0].item())
    lam_ols = max(0.0, float(-a))
    lam_max = max(5.0, 5.0*(lam_ols + 1.0))

    lambdas = torch.linspace(0.0, lam_max, n_grid, device=V_fit.device, dtype=V_fit.dtype)
    rhs = (LV_fit + margin_sigma*SE_fit).unsqueeze(0) + lambdas.unsqueeze(1) * V_fit.unsqueeze(0)
    b_vals = rhs.max(dim=1).values
    j = int(torch.argmin(b_vals).item())
    return float(lambdas[j].item()), float(b_vals[j].item())

@torch.no_grad()
def holdout_report(V_te, LV_te, SE_te, lam, b, ksig: float):
    diff = (LV_te + ksig*SE_te) + lam*V_te - b
    viol = diff > 0.0
    return dict(
        violations=int(viol.sum().item()),
        total=int(V_te.numel()),
        frac=float(viol.float().mean().item()),
        worst=float(diff.max().item()),
    )

# ============================================================
# Main
# ============================================================

@torch.no_grad()
def run():
    L = int(CFG["L"])
    K_total = int(CFG["K_total"])
    K_fit = int(CFG["K_fit"])

    eps = float(CFG["eps_fd"])
    mc = int(CFG["mc"])
    mc_chunk = int(CFG["mc_chunk"])
    batch_cfg = int(CFG["batch_cfg"])

    beta_list = [float(b) for b in CFG["beta_list"]]
    n_grid = int(CFG["n_grid"])

    meta = make_meta(K_total, CFG["sigma_list"], CFG["frac_haar"], CFG["seed"])

    # storage on CPU
    Vbar_all = np.empty((K_total,), dtype=np.float64)
    Bavg_all = np.empty((K_total,), dtype=np.float64)
    lap_all  = np.empty((K_total,), dtype=np.float64)
    lap_se_all = np.empty((K_total,), dtype=np.float64)

    LV_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    LV_se_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    gip_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}
    gip_se_all = {beta: np.empty((K_total,), dtype=np.float64) for beta in beta_list}

    t0 = time.time()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    for start in range(0, K_total, batch_cfg):
        end = min(K_total, start + batch_cfg)
        mb = meta[start:end]
        U = gen_U_batch(mb, L=L)

        # unperturbed stats
        z0, Bavg0, V0 = zsum_Bavg_Vbar(U)
        B = U.shape[0]

        # accumulators
        sum_lap  = torch.zeros((B,), device=device, dtype=torch.float64)
        sum_lap2 = torch.zeros((B,), device=device, dtype=torch.float64)

        sum_gip  = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_gip2 = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_LV   = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}
        sum_LV2  = {beta: torch.zeros((B,), device=device, dtype=torch.float64) for beta in beta_list}

        done = 0
        while done < mc:
            k = min(mc_chunk, mc - done)

            # Xi: (k,B,L,L,L,L,4,3)
            Xi = torch.randn((k, B, L, L, L, L, 4, 3), device=device, dtype=Xi_dtype)

            exp_p = su2_exp(eps * Xi)   # (k,B,...,4)
            exp_m = su2_conj(exp_p)

            Up = su2_mul(U.unsqueeze(0), exp_p).reshape(k*B, L, L, L, L, 4, 4)
            Um = su2_mul(U.unsqueeze(0), exp_m).reshape(k*B, L, L, L, L, 4, 4)

            z_p, _, V_p = zsum_Bavg_Vbar(Up)
            z_m, _, V_m = zsum_Bavg_Vbar(Um)

            z_p = z_p.view(k, B)
            z_m = z_m.view(k, B)
            V_p = V_p.view(k, B)
            V_m = V_m.view(k, B)

            lap = (V_p + V_m - 2.0*V0.unsqueeze(0)) / (eps*eps)    # (k,B)
            dV  = (V_p - V_m) / (2.0*eps)                          # (k,B)
            dz  = (z_p - z_m) / (2.0*eps)                          # (k,B)  beta-free

            sum_lap  += lap.sum(dim=0).to(torch.float64)
            sum_lap2 += (lap*lap).sum(dim=0).to(torch.float64)

            for beta in beta_list:
                dS = beta * dz
                gip = dS * dV
                LV  = lap - gip

                sum_gip[beta]  += gip.sum(dim=0).to(torch.float64)
                sum_gip2[beta] += (gip*gip).sum(dim=0).to(torch.float64)
                sum_LV[beta]   += LV.sum(dim=0).to(torch.float64)
                sum_LV2[beta]  += (LV*LV).sum(dim=0).to(torch.float64)

            done += k
            del Xi, exp_p, exp_m, Up, Um, z_p, z_m, V_p, V_m, lap, dV, dz
            if device.type == "cuda":
                torch.cuda.empty_cache()

        lap_m, lap_se = mean_se(sum_lap, sum_lap2, mc)

        Vbar_all[start:end] = V0.detach().cpu().numpy()
        Bavg_all[start:end] = Bavg0.detach().cpu().numpy()
        lap_all[start:end]  = lap_m.detach().cpu().numpy()
        lap_se_all[start:end] = lap_se.detach().cpu().numpy()

        for beta in beta_list:
            gip_m, gip_se = mean_se(sum_gip[beta], sum_gip2[beta], mc)
            LV_m, LV_se   = mean_se(sum_LV[beta],  sum_LV2[beta],  mc)

            gip_all[beta][start:end] = gip_m.detach().cpu().numpy()
            gip_se_all[beta][start:end] = gip_se.detach().cpu().numpy()
            LV_all[beta][start:end] = LV_m.detach().cpu().numpy()
            LV_se_all[beta][start:end] = LV_se.detach().cpu().numpy()

        if device.type == "cuda":
            torch.cuda.empty_cache()

        rate = end / max(1e-9, (time.time() - t0))
        print(f"[progress] {end}/{K_total} configs  ({rate:.2f} cfg/s)", flush=True)

    # ----------------------------
    # Fit + holdout per beta
    # ----------------------------
    print("\n============================")
    print("BETA SWEEP — FIT + HOLDOUT")
    print("============================")

    V = torch.tensor(Vbar_all, dtype=torch.float64)

    for beta in beta_list:
        LV = torch.tensor(LV_all[beta], dtype=torch.float64)
        SE = torch.tensor(LV_se_all[beta], dtype=torch.float64)

        V_fit, LV_fit, SE_fit = V[:K_fit], LV[:K_fit], SE[:K_fit]
        V_te,  LV_te,  SE_te  = V[K_fit:], LV[K_fit:], SE[K_fit:]

        lam0, b0 = fit_drift_one_sided(V_fit, LV_fit, SE_fit, margin_sigma=0.0, n_grid=n_grid)
        lam2, b2 = fit_drift_one_sided(V_fit, LV_fit, SE_fit, margin_sigma=2.0, n_grid=n_grid)

        rep0 = holdout_report(V_te, LV_te, SE_te, lam0, b0, ksig=0.0)
        rep2 = holdout_report(V_te, LV_te, SE_te, lam2, b2, ksig=2.0)

        print(f"\n[beta={beta:.6g}]")
        print(f"  fit@0σ: lambda={lam0:.6g}  b={b0:.6g}  holdout viol={rep0['violations']}/{rep0['total']}  worst={rep0['worst']:.3g}")
        print(f"  fit@2σ: lambda={lam2:.6g}  b={b2:.6g}  holdout viol={rep2['violations']}/{rep2['total']}  worst={rep2['worst']:.3g}")

    # ----------------------------
    # Save NPZ
    # ----------------------------
    out = dict(
        L=np.full((K_total,), CFG["L"], dtype=np.int32),
        Vbar=Vbar_all,
        Bavg=Bavg_all,
        lap=lap_all,
        lap_se=lap_se_all,
        K_fit=np.int32(K_fit),
        beta_list=np.array(beta_list, dtype=np.float64),
    )
    for beta in beta_list:
        tag = f"b{beta:.6g}".replace(".", "p")
        out[f"LV_{tag}"] = LV_all[beta]
        out[f"LVse_{tag}"] = LV_se_all[beta]
        out[f"gip_{tag}"] = gip_all[beta]
        out[f"gipse_{tag}"] = gip_se_all[beta]

    np.savez_compressed(CFG["out_npz"], **out)
    print(f"\n[saved] {CFG['out_npz']}", flush=True)

    if device.type == "cuda":
        peak = torch.cuda.max_memory_allocated() / 2**30
        print(f"[CUDA] peak allocated: {peak:.2f} GB", flush=True)

if __name__ == "__main__":
    run()


[device] cuda NVIDIA L4
[progress] 64/2048 configs  (0.51 cfg/s)
[progress] 128/2048 configs  (0.51 cfg/s)
[progress] 192/2048 configs  (0.51 cfg/s)
[progress] 256/2048 configs  (0.51 cfg/s)
[progress] 320/2048 configs  (0.51 cfg/s)
[progress] 384/2048 configs  (0.51 cfg/s)
[progress] 448/2048 configs  (0.51 cfg/s)
[progress] 512/2048 configs  (0.51 cfg/s)
[progress] 576/2048 configs  (0.51 cfg/s)
[progress] 640/2048 configs  (0.51 cfg/s)
[progress] 704/2048 configs  (0.51 cfg/s)
[progress] 768/2048 configs  (0.51 cfg/s)
[progress] 832/2048 configs  (0.51 cfg/s)
[progress] 896/2048 configs  (0.51 cfg/s)
[progress] 960/2048 configs  (0.51 cfg/s)
[progress] 1024/2048 configs  (0.51 cfg/s)
[progress] 1088/2048 configs  (0.51 cfg/s)
[progress] 1152/2048 configs  (0.51 cfg/s)
[progress] 1216/2048 configs  (0.51 cfg/s)
[progress] 1280/2048 configs  (0.51 cfg/s)
[progress] 1344/2048 configs  (0.51 cfg/s)
[progress] 1408/2048 configs  (0.51 cfg/s)
[progress] 1472/2048 configs  (0.51 cfg/s)
[pr